# YGOPRODeck Banlist Scraper (TCG & OCG)

This notebook scrapes YGOPRODeck banlists via their public endpoints referenced in the site HTML:
- Fetches all available banlist dates for TCG and OCG
- For each format/date, fetches all cards and their classification (Forbidden, Limited, Semi-Limited)
- Builds a pandas DataFrame with columns: format, date, card_name, status, card_id, pretty_url, type
- Optionally saves to `Data/banlist_history_YGOPRO.csv`

Note: The page HTML loads content via AJAX; we call the same API endpoints directly with a retry-friendly HTTP session.

In [1]:
import time
from typing import List, Dict, Any
import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

BANLIST_DATES_URL = "https://ygoprodeck.com/api/banlist/getBanListDates.php"
BANLIST_URL = "https://ygoprodeck.com/api/banlist/getBanList.php"
VALID_LISTS = {"TCG", "OCG"}


def build_session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    adapter = HTTPAdapter(max_retries=retries)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": "https://ygoprodeck.com/banlist/",
    })
    return s

session = build_session()

In [2]:
def get_all_dates(session: requests.Session) -> List[Dict[str, Any]]:
    resp = session.get(BANLIST_DATES_URL, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    # Expect list of {"type": "TCG", "date": "YYYY-MM-DD"}
    assert isinstance(data, list), "Unexpected dates payload"
    # sort by date asc within type for stable processing
    for_type = {}
    for item in data:
        for_type.setdefault(item.get("type"), []).append(item)
    for t in for_type:
        for_type[t] = sorted(for_type[t], key=lambda x: x["date"])  # oldest->newest
    # flatten back preserving global order grouped by type
    ordered = []
    for t in ("TCG", "OCG"):
        if t in for_type:
            ordered.extend(for_type[t])
    return ordered


def fetch_banlist(session: requests.Session, list_type: str, date: str) -> List[Dict[str, Any]]:
    params = {"list": list_type, "date": date}
    resp = session.get(BANLIST_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    # Expect list of items with keys: id, name, pretty_url, type, status_text
    assert isinstance(data, list), f"Unexpected banlist payload for {list_type} {date}"
    return data

In [3]:
records = []

all_dates = get_all_dates(session)

# Filter to TCG/OCG only
work_dates = [d for d in all_dates if d.get("type") in VALID_LISTS]
print(f"Found {len(work_dates)} date entries across TCG/OCG.")

for i, item in enumerate(work_dates, 1):
    fmt = item["type"]
    dt = item["date"]
    try:
        entries = fetch_banlist(session, fmt, dt)
    except Exception as e:
        print(f"[{i}/{len(work_dates)}] {fmt} {dt}: ERROR {e}")
        continue

    for c in entries:
        status = c.get("status_text")
        if status not in ("Forbidden", "Limited", "Semi-Limited"):
            # Ignore other statuses if any
            continue
        records.append({
            "format": fmt,
            "date": dt,
            "card_name": c.get("name"),
            "status": status,
            "card_id": c.get("id"),
            "pretty_url": c.get("pretty_url"),
            "type": c.get("type"),
        })
    print(f"[{i}/{len(work_dates)}] {fmt} {dt}: {len(entries)} items")
    time.sleep(0.15)  # polite pacing

banlist_df = pd.DataFrame.from_records(records)
banlist_df.sort_values(["format", "date", "status", "card_name"], inplace=True)
banlist_df.reset_index(drop=True, inplace=True)
banlist_df.head()

Found 147 date entries across TCG/OCG.
[1/147] TCG 1999-08-01: 3 items
[2/147] TCG 2000-04-01: 14 items
[3/147] TCG 2000-11-01: 22 items
[4/147] TCG 2001-05-01: 39 items
[5/147] TCG 2003-01-01: 43 items
[6/147] TCG 2003-09-01: 57 items
[7/147] TCG 2004-03-01: 59 items
[8/147] TCG 2004-09-01: 58 items
[9/147] TCG 2005-03-01: 77 items
[10/147] TCG 2005-09-01: 80 items
[11/147] TCG 2006-03-01: 92 items
[12/147] TCG 2006-09-01: 95 items
[13/147] TCG 2007-03-01: 100 items
[14/147] TCG 2007-09-01: 110 items
[15/147] TCG 2008-03-01: 109 items
[16/147] TCG 2008-09-01: 113 items
[17/147] TCG 2009-03-01: 119 items
[18/147] TCG 2009-09-01: 125 items
[19/147] TCG 2010-03-01: 135 items
[20/147] TCG 2010-09-01: 132 items
[21/147] TCG 2011-03-01: 134 items
[22/147] TCG 2011-09-01: 135 items
[23/147] TCG 2012-03-01: 140 items
[24/147] TCG 2012-09-01: 145 items
[25/147] TCG 2013-03-01: 143 items
[26/147] TCG 2013-09-01: 155 items
[27/147] TCG 2013-10-11: 155 items
[28/147] TCG 2014-01-01: 158 items
[29

,format,date,card_name,status,card_id,pretty_url,type
0,OCG,1999-07-01,Dark Hole,Limited,53129443,dark-hole-4525,None
1,OCG,1999-07-01,Raigeki,Limited,12580477,raigeki-1087,None
2,OCG,1999-07-01,Trap Hole,Limited,4206964,trap-hole-379,None
3,OCG,2000-02-01,Change of Heart,Limited,4031928,change-of-heart-359,None
4,OCG,2000-02-01,Dark Hole,Limited,53129443,dark-hole-4525,None


In [4]:
# Save to CSV (YGOPRO source)
out_path = r"c:\\Users\\mlchr\\OneDrive\\Personal\\Projects\\heart of the cards\\Data\\banlist_history_YGOPRO.csv"
banlist_df.to_csv(out_path, index=False)
print(f"Saved {len(banlist_df):,} rows to {out_path}")

Saved 20,098 rows to c:\\Users\\mlchr\\OneDrive\\Personal\\Projects\\heart of the cards\\Data\\banlist_history_YGOPRO.csv
